# Analisi del Redshift con dati SDSS

## Introduzione

Il **redshift** (spostamento verso il rosso) è un fenomeno fondamentale in astronomia e cosmologia. Si verifica quando la luce emessa da un oggetto celeste si sposta verso lunghezze d'onda maggiori (verso il rosso) a causa dell'espansione dell'Universo. Valori di redshift più elevati indicano oggetti più lontani e osservati in epoche più antiche dell'Universo.

Lo **Sloan Digital Sky Survey (SDSS)** è uno dei più importanti survey astronomici, che ha catalogato centinaia di milioni di oggetti celesti fornendo dati spettroscopici e fotometrici in cinque bande (u, g, r, i, z).

In questo notebook:
1. Eseguiamo una query al database SDSS per ottenere 100 oggetti con redshift misurato
2. Visualizziamo la distribuzione del redshift
3. Filtriamo per tipo di oggetto (quasar)
4. Standardizziamo i dati di redshift con Z-score

In [ ]:
from astroquery.sdss import SDSS
from astropy.table import Table
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Query al database SDSS

Eseguiamo una query SQL per ottenere 100 oggetti con redshift tra 0.01 e 1. Uniamo la tabella `PhotoObjAll` (dati fotometrici) con `SpecObj` (dati spettroscopici) per ottenere sia la classificazione che il redshift.

In [ ]:
query = """
SELECT TOP 100
    p.objid, p.ra, p.dec, s.z as redshift, p.type, p.u, p.g, p.r, p.i, p.z
FROM
    PhotoObjAll AS p
JOIN
    SpecObj AS s ON s.bestobjid = p.objid
WHERE
    s.zWarning = 0
AND
    s.z BETWEEN 0.01 AND 1
ORDER BY
    s.z DESC
"""

result = SDSS.query_sql(query)
df = result.to_pandas()
df.to_csv("redshift_sdss_data.csv", index=False)
df.head()

## 2. Esplorazione dei dati

Il campo `type` in SDSS codifica il tipo di oggetto:
- 0: Sconosciuto / Non classificabile
- 1: Stella
- 2: Galassia
- 3: Quasar (QSO)
- 6: Cielo (porzione di cielo senza oggetto)

In [ ]:
df.info()

In [ ]:
print("Distribuzione dei tipi di oggetto:")
print(df['type'].value_counts().sort_index())

type_labels = {0: 'Unknown', 1: 'Star', 2: 'Galaxy', 3: 'QSO', 6: 'Sky'}
df['type_label'] = df['type'].map(type_labels)

## 3. Distribuzione del Redshift

Visualizziamo la distribuzione del redshift per tutti gli oggetti.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['redshift'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
plt.xlabel('Redshift')
plt.ylabel('Numero di oggetti')
plt.title('Distribuzione del Redshift (dati SDSS)')
plt.grid(alpha=0.3)
plt.show()

## 4. Filtraggio per tipo: Quasar (QSO)

Filtriamo i dati per selezionare solo i **Quasar** (type = 3). I quasar sono nuclei galattici attivi estremamente luminosi e rappresentano alcune delle galassie più distanti osservabili.

In [ ]:
df_qso = df[df['type'] == 3].copy()
print(f"Numero di QSO trovati: {len(df_qso)}")
df_qso.head()

## 5. Standardizzazione (Z-score)

Applichiamo una standardizzazione Z-score al redshift dei QSO:

$z_{std} = \frac{z - \mu}{\sigma}$

dove $\mu$ è la media e $\sigma$ è la deviazione standard. Questo trasforma i dati in una distribuzione con media 0 e deviazione standard 1.

In [ ]:
df_qso.loc[:, 'redshift_standardized'] = (
    (df_qso['redshift'] - df_qso['redshift'].mean()) / df_qso['redshift'].std()
)
df_qso[['redshift', 'redshift_standardized']].head(10)

## 6. Visualizzazione dei risultati

Confrontiamo il redshift originale con quello standardizzato.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].plot(df_qso['redshift'], marker='o', linestyle='-', markersize=4, color='steelblue')
axes[0].set_title('Redshift Originale')
axes[0].set_xlabel('Indice')
axes[0].set_ylabel('Redshift')
axes[0].grid(alpha=0.3)

axes[1].scatter(
    range(len(df_qso['redshift_standardized'])),
    df_qso['redshift_standardized'],
    color='coral', marker='o', alpha=0.7
)
axes[1].set_title('Redshift Standardizzato (Z-score)')
axes[1].set_xlabel('Indice')
axes[1].set_ylabel('Redshift Standardizzato')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusioni

Abbiamo visto come:
- Eseguire query SQL sul database SDSS per ottenere dati astronomici
- Esplorare la distribuzione del redshift per diversi tipi di oggetti
- Filtrare i dati per tipo di oggetto (QSO)
- Applicare una standardizzazione Z-score ai dati di redshift

Il redshift è uno strumento fondamentale per misurare le distanze cosmiche e studiare l'evoluzione dell'Universo.